# Global timing jitter

Simulation, thermodynamic diagnostics, persistent homology, persistence images and silhouettes, classification, and the publication figures for the global timing-jitter benchmark.

The notebook uses the parameter choices reported in the revised manuscript. Full runs use 1000 trajectories and may require substantial computation time.


In [ ]:



import warnings
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, skew, kurtosis, entropy

from ripser import ripser
from persim import wasserstein, bottleneck

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve


plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",  
    "font.size": 10,             
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": False,     
    "axes.linewidth": 1.0,       
    "lines.linewidth": 1.5,
    "lines.markersize": 4,
    "xtick.direction": "in",     
    "ytick.direction": "in",
    "xtick.top": True,           
    "ytick.right": True,
    "figure.dpi": 300,           
    "savefig.dpi": 300,
    "savefig.bbox": "tight"      
})

In [ ]:



def rotate_bloch(r, omega_vec, dt):
    """Unitary Bloch rotation: r_dot = omega x r"""
    return r + dt * np.cross(omega_vec, r)


def thermalize_bloch(r, omega_z, T, Gamma, dt):
    """Markovian thermalization toward Gibbs state."""
    beta = 1.0 / max(T, 1e-9)
    z_eq = -np.tanh(0.5 * beta * omega_z)
    x, y, z = r
    x += dt * (-0.5 * Gamma * x)
    y += dt * (-0.5 * Gamma * y)
    z += dt * (-Gamma * (z - z_eq))
    return np.array([x, y, z], dtype=float)


def simulate_otto(
    Th=0.8, Tc=0.25,
    omega_h=2.0, omega_c=1.0,
    omega_x_max=1.0,
    Gamma=0.6,
    tau_h=0.7, tau_1=0.6, tau_c=0.7, tau_3=0.6,
    sigma_tau=0.0,
    burn_in=2, M=5,
    steps=40,
    seed=0
):
    """
    Returns:
    r_traj : Bloch vectors along *full* trajectory (includes transient burn-in)
    work_cycles : cycle work estimate for cycles AFTER burn_in (length M)
    """
    rng = np.random.default_rng(seed)
    r = np.array([0.0, 0.0, -0.2], dtype=float)  # Initial Bloch state

    r_list = []
    work_cycles = []

    def jitter(tau):
        return max(1e-4, tau * (1 + rng.normal(0.0, sigma_tau)))

    for cyc in range(burn_in + M):
        th, t1, tc, t3 = map(jitter, [tau_h, tau_1, tau_c, tau_3])
        W = 0.0

        # Hot isochore
        dt = th / steps
        for _ in range(steps):
            r = thermalize_bloch(r, omega_h, Th, Gamma, dt)
            r_list.append(r.copy())

        # Expansion stroke
        dt = t1 / steps
        E_prev = 0.5 * omega_h * r[2]
        for k in range(steps):
            s = (k + 0.5) / steps
            omega_z = omega_h + (omega_c - omega_h) * s
            omega_x = omega_x_max * np.sin(np.pi * s)
            r = rotate_bloch(r, (omega_x, 0.0, omega_z), dt)
            E = 0.5 * (omega_z * r[2] + omega_x * r[0])
            W += E - E_prev
            E_prev = E
            r_list.append(r.copy())

        # Cold isochore
        dt = tc / steps
        for _ in range(steps):
            r = thermalize_bloch(r, omega_c, Tc, Gamma, dt)
            r_list.append(r.copy())

        # Compression stroke
        dt = t3 / steps
        E_prev = 0.5 * omega_c * r[2]
        for k in range(steps):
            s = (k + 0.5) / steps
            omega_z = omega_c + (omega_h - omega_c) * s
            omega_x = omega_x_max * np.sin(np.pi * s)
            r = rotate_bloch(r, (omega_x, 0.0, omega_z), dt)
            E = 0.5 * (omega_z * r[2] + omega_x * r[0])
            W += E - E_prev
            E_prev = E
            r_list.append(r.copy())

        if cyc >= burn_in:
            work_cycles.append(W)

    return np.array(r_list, dtype=float), np.array(work_cycles, dtype=float)

In [ ]:



def takens_embed(x, d=3, tau=10):
    x = np.asarray(x, dtype=float)
    n = len(x) - (d - 1) * tau
    if n <= 0:
        raise ValueError("Time series too short for embedding.")
    return np.column_stack([x[i:i + n] for i in range(0, d * tau, tau)])


def ph_diagram_H1(P):
    out = ripser(P, maxdim=1)  # Omit `thresh=None` for Python 3.12 compatibility
    H1 = out["dgms"][1] if len(out["dgms"]) > 1 else np.zeros((0, 2))
    H1 = H1[np.isfinite(H1[:, 1])]
    return H1


def diagram_distances(dgm, dgm_ref):
    if len(dgm) == 0 and len(dgm_ref) == 0:
        return 0.0, 0.0
    W = float(wasserstein(dgm, dgm_ref, matching=False))
    B = float(bottleneck(dgm, dgm_ref))
    return W, B


def persistence_image(dgm, resolution=(40, 40), sigma=0.02, max_b=0.8, max_p=0.8):
    """Zoomed-in canvas with a smaller blur radius (sigma) to see fine details"""
    if len(dgm) == 0:
        return np.zeros(resolution[0] * resolution[1])
    
    births = dgm[:, 0]
    pers = dgm[:, 1] - dgm[:, 0]
    
    bx = np.linspace(0.0, max_b, resolution[0])
    py = np.linspace(0.0, max_p, resolution[1])
    B, P = np.meshgrid(bx, py, indexing="ij")
    
    img = np.zeros_like(B)
    for b, p in zip(births, pers):
        img += p * np.exp(-((B - b)**2 + (P - p)**2) / (2 * sigma**2))
        
    return img.flatten() 

def persistence_silhouette(dgm, resolution=100, max_grid=0.8):
    """Zoomed-in x-axis to catch the micro-structures"""
    if len(dgm) == 0:
        return np.zeros(resolution)
        
    grid = np.linspace(0.0, max_grid, resolution)
    silhouette = np.zeros_like(grid)
    
    for b, d in dgm:
        p = d - b
        mid = (b + d) / 2.0
        half_p = p / 2.0
        triangle = np.maximum(0, half_p - np.abs(grid - mid))
        # Square-root persistence weighting balances dominant and low-persistence features
        silhouette += np.sqrt(p) * triangle 
        
    return silhouette


def extract_baseline_features(x):
    
    std_dev = np.std(x)
    skewness = skew(x)
    kurt = kurtosis(x)
    p2p = np.max(x) - np.min(x)
    rms = np.sqrt(np.mean(x**2))
    
    
    fft_vals = np.abs(np.fft.rfft(x))
    freqs = np.fft.rfftfreq(len(x))
    # Guard against a zero spectral denominator
    if np.sum(fft_vals) == 0:
        spectral_centroid = 0.0
    else:
        spectral_centroid=np.sum(freqs*fft_vals)/np.sum(fft_vals)
        
    return [std_dev, skewness, kurt, p2p, rms, spectral_centroid]

In [ ]:



def steady_slice_indices(burn_in, steps):
    """
    Each cycle appends 4*steps points to r_list:
    hot (steps) + expansion (steps) + cold (steps) + compression (steps).
    We remove burn_in cycles worth of points for TDA/QI to ensure steady state.
    """
    points_per_cycle = 4 * int(steps)
    return int(burn_in) * points_per_cycle


def build_hybrid_dataset(
    sigma_list, n_runs_per_sigma=135, nominal_max_sigma=0.05,
    degraded_min_sigma=0.15, embed_dim=3, embed_tau=10, seed_base=1000, **params
):
    X_dist, X_img, X_sil, X_base, y, sig_used = [], [], [], [], [], []

    # Build the nominal reference persistence diagram
    r_ref, _ = simulate_otto(sigma_tau=0.0, seed=999, **params)
    cut = steady_slice_indices(params["burn_in"], params["steps"])
    series_ref = r_ref[cut:, 0]
    P_ref = takens_embed(series_ref, d=embed_dim, tau=embed_tau)
    dgm_ref = ph_diagram_H1(P_ref)

    pbar = tqdm(total=len(sigma_list) * n_runs_per_sigma, desc="Building Hybrid Dataset", unit="run")

    for s in sigma_list:
        s = float(s)
        for k in range(n_runs_per_sigma):
            seed = seed_base + 31 * k + int(1000 * s)
            r, _ = simulate_otto(sigma_tau=s, seed=seed, **params)

            
            series_clean = r[cut:, 0]
            
            # Add Gaussian readout noise
            
            series = r[cut:, 0]
            
            P = takens_embed(series, d=embed_dim, tau=embed_tau)
            dgm = ph_diagram_H1(P)

            
            Wd, Bd = diagram_distances(dgm, dgm_ref)
            X_dist.append([Wd + Bd, Wd, Bd])
            
            
            X_img.append(persistence_image(dgm, resolution=(40, 40), sigma=0.02, max_b=0.8, max_p=0.8))
            X_sil.append(persistence_silhouette(dgm, resolution=100, max_grid=0.8))
            
            
            X_base.append(extract_baseline_features(series))
            
            sig_used.append(s)
            # Assign labels using the fixed operational threshold
            if s <= nominal_max_sigma:
                y.append(0)
            elif s >= degraded_min_sigma:
                y.append(1)
            else:
                y.append(None)

            pbar.update(1)

    pbar.close()

    
    mask = np.array([yi is not None for yi in y])
    return (np.array(X_dist)[mask], np.array(X_img)[mask], np.array(X_sil)[mask], np.array(X_base)[mask],
            np.array(y)[mask].astype(int), np.array(sig_used)[mask])

In [ ]:



FAST_PARAMS = dict(
    Th=0.8, Tc=0.25,
    omega_h=2.0, omega_c=1.0, omega_x_max=1.0,
    Gamma=0.6,
    tau_h=0.7, tau_1=0.6, tau_c=0.7, tau_3=0.6,
    burn_in=2,  
    M=5,        
    steps=40    
)


PLOT_PARAMS = FAST_PARAMS.copy()
PLOT_PARAMS["burn_in"] = 15  
PLOT_PARAMS["M"] = 15        


n_total_runs = 1000
np.random.seed(147)
continuous_sigma_list = np.random.uniform(0.0, 0.25, n_total_runs)


X_dist, X_img, X_sil, X_base, y, sig_used = build_hybrid_dataset(
    continuous_sigma_list, 
    n_runs_per_sigma=1,            
    nominal_max_sigma=0.125,       
    degraded_min_sigma=0.12500001, 
    seed_base=2000,
    **FAST_PARAMS
)


print("Class distribution:", np.unique(y, return_counts=True))


QI = X_dist[:, 0]
W1 = X_dist[:, 1]
B  = X_dist[:, 2]

print(f"Distances shape: {X_dist.shape}")
print(f"Images shape: {X_img.shape}")
print(f"Silhouettes shape: {X_sil.shape}")
print(f"Baselines shape: {X_base.shape}")

In [ ]:



def plot_bloch_and_delay(params, sigma, seed=1, d=3, tau=10):
    r, _ = simulate_otto(sigma_tau=float(sigma), seed=seed, **params)

    color = "#003366" if sigma == 0.0 else "#8B0000"
    label = r"Nominal ($\sigma_\tau = 0$)" if sigma == 0.0 else rf"Degraded ($\sigma_\tau = {sigma:.2f}$)"

    fig = plt.figure(figsize=(10.0, 5.0))

    
    ax1 = fig.add_subplot(121, projection="3d")
    
    ax1.plot(r[:, 0], r[:, 1], r[:, 2], color=color, linewidth=0.3, alpha=0.4)
    ax1.scatter(r[:, 0], r[:, 1], r[:, 2], s=3, c=color, alpha=0.3, edgecolor='none')
    
    ax1.set_xlabel(r"$\langle\sigma_x\rangle$", fontsize=16, labelpad=8)
    ax1.set_ylabel(r"$\langle\sigma_y\rangle$", fontsize=16, labelpad=8)
    ax1.set_zlabel(r"$\langle\sigma_z\rangle$", fontsize=16, labelpad=8)

    
    ax1_zticks = [-0.8, -0.6, -0.4, -0.2, 0.0]
    ax1.set_zticks(ax1_zticks)

    ax1.tick_params(axis='x', which='major', labelsize=14, pad=2)
    ax1.tick_params(axis='y', which='major', labelsize=14, pad=2)
    ax1.tick_params(axis='z', which='major', labelsize=14, pad=6) 

    
    P = takens_embed(r[:, 0], d=d, tau=tau)
    ax2 = fig.add_subplot(122, projection="3d")
    
    ax2.plot(P[:, 0], P[:, 1], P[:, 2], color=color, linewidth=0.3, alpha=0.4)
    ax2.scatter(P[:, 0], P[:, 1], P[:, 2], s=3, c=color, alpha=0.3, edgecolor='none')
    
    ax2.set_xlabel(r"$x(t)$", fontsize=16, labelpad=8)
    ax2.set_ylabel(r"$x(t+\tau)$", fontsize=16, labelpad=8)
    
    

    
    ax2_ticks = [-0.4, -0.2, 0.0]
    ax2.set_xticks(ax2_ticks)
    ax2.set_yticks(ax2_ticks)
    ax2.set_zticks(ax2_ticks)

    ax2.tick_params(axis='x', which='major', labelsize=14, pad=2)
    ax2.tick_params(axis='y', which='major', labelsize=14, pad=2)
    ax2.tick_params(axis='z', which='major', labelsize=14, pad=6)  
    
    
    fig.subplots_adjust(left=0.05, right=0.85, bottom=0.1, top=0.95, wspace=0.2)
    
    
    fig.text(0.9, 0.5, r"$x(t+2\tau)$", va='center', ha='center', rotation='vertical', fontsize=16)

    plt.show()


plot_bloch_and_delay(PLOT_PARAMS, sigma=0.0, seed=1, d=3, tau=10)
plot_bloch_and_delay(PLOT_PARAMS, sigma=0.25, seed=1, d=3, tau=10)

In [ ]:



def plot_work_per_cycle(params, sigma_a=0.0, sigma_b=0.25, seed_a=1, seed_b=2):
    _, Wa = simulate_otto(sigma_tau=float(sigma_a), seed=seed_a, **params)
    _, Wb = simulate_otto(sigma_tau=float(sigma_b), seed=seed_b, **params)

    fig, ax = plt.subplots(figsize=(3.4, 2.8))
    
    ax.plot(np.arange(len(Wa)), Wa, "o-", label=rf"$\sigma_\tau={sigma_a:.2f}$", color="#003366", markersize=4)
    ax.plot(np.arange(len(Wb)), Wb, "s-", label=rf"$\sigma_\tau={sigma_b:.2f}$", color="#8B0000", markersize=4)
    
    ax.set_xlabel("Cycle index")
    ax.set_ylabel(r"Cycle work $W_n$ (arb.)")
    ax.legend()
    
    plt.tight_layout()
    plt.show()


plot_work_per_cycle(PLOT_PARAMS, sigma_a=0.0, sigma_b=0.25, seed_a=1, seed_b=2)

def work_diagnostics(params, sigma_list, n_runs=15, seed_base=3000):
    meanW, varW, sigs = [], [], []
    for s in sigma_list:
        s = float(s)
        for k in range(n_runs):
            seed = seed_base + 17 * k + int(1000 * s)
            _, Wc = simulate_otto(sigma_tau=s, seed=seed, **params)
            if len(Wc) > 0:
                meanW.append(float(np.mean(Wc)))
                varW.append(float(np.var(Wc)))
            else:
                meanW.append(0.0)
                varW.append(0.0)
            sigs.append(s)
    return np.array(meanW), np.array(varW), np.array(sigs)

sigma_grid = np.linspace(0.0, 0.25, 20)


meanW, varW, work_sig = work_diagnostics(PLOT_PARAMS, sigma_grid, n_runs=15)


fig, ax = plt.subplots(figsize=(3.4, 2.8))
for s in np.unique(work_sig):
    vals = meanW[work_sig == s]
    ax.scatter(np.full_like(vals, s, dtype=float), vals, s=15, alpha=0.6, color="#003366", edgecolor='none')

ax.plot(
    np.unique(work_sig),
    [np.mean(meanW[work_sig == s]) for s in np.unique(work_sig)],
    marker="o", color="black", markersize=4, linewidth=1.5
)
ax.set_xlabel(r"Timing Jitter $\sigma_\tau$")
ax.set_ylabel(r"Mean Cycle Work $\bar{W}$ (arb.)")
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(3.4, 2.8))
for s in np.unique(work_sig):
    vals = varW[work_sig == s]
    ax.scatter(np.full_like(vals, s, dtype=float), vals, s=15, alpha=0.6, color="#8B0000", edgecolor='none')

ax.plot(
    np.unique(work_sig),
    [np.mean(varW[work_sig == s]) for s in np.unique(work_sig)],
    marker="o", color="black", markersize=4, linewidth=1.5
)
ax.set_xlabel(r"Timing Jitter $\sigma_\tau$")
ax.set_ylabel(r"Work Variance $\mathrm{Var}_W(\sigma_\tau)$")
plt.tight_layout()
plt.show()

In [ ]:



fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.6))
names = ["QI", "Wasserstein", "Bottleneck"]
feature_arrays = [QI, W1, B]

colors = ['#003366', '#8B0000'] 

for j in range(3):
    axs[j].hist(feature_arrays[j][y == 0], bins=12, alpha=0.7, label="Nominal", color=colors[0], density=True)
    axs[j].hist(feature_arrays[j][y == 1], bins=12, alpha=0.7, label="Degraded", color=colors[1], density=True)
    
    axs[j].set_xlabel(names[j])
    if j == 0:
        axs[j].set_ylabel("Probability Density")
    if j == 2:
        axs[j].legend(loc='upper right')

plt.tight_layout()
plt.show()


r_qi, p_qi = pearsonr(sig_used, QI)
print(f"Pearson correlation (QI vs σ_tau): r = {r_qi:.3f}, p = {p_qi:.2e}")

coef = np.polyfit(sig_used, QI, 1)
xx = np.linspace(sig_used.min(), sig_used.max(), 200)
yy = coef[0] * xx + coef[1]

fig, ax = plt.subplots(figsize=(3.4, 2.8))


ax.scatter(sig_used, QI, s=15, alpha=0.6, color="#003366", edgecolor='none')
ax.plot(xx, yy, linestyle="--", linewidth=1.5, color='black', label=rf"Linear fit ($r={r_qi:.3f}$)")

ax.set_xlabel(r"Timing Jitter $\sigma_\tau$")
ax.set_ylabel("QI")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:



r_nom, _ = simulate_otto(sigma_tau=0.0, seed=1, **FAST_PARAMS)
cut = steady_slice_indices(FAST_PARAMS["burn_in"], FAST_PARAMS["steps"])
P_nom = takens_embed(r_nom[cut:, 0], d=3, tau=10)
dgm_nominal = ph_diagram_H1(P_nom)

r_deg, _ = simulate_otto(sigma_tau=0.25, seed=2, **FAST_PARAMS)
P_deg = takens_embed(r_deg[cut:, 0], d=3, tau=10)
dgm_degraded = ph_diagram_H1(P_deg)


def get_ranges(dgm1, dgm2):
    births = np.concatenate([dgm1[:, 0], dgm2[:, 0]]) if len(dgm1) and len(dgm2) else \
             (dgm1[:, 0] if len(dgm1) else dgm2[:, 0])
    deaths = np.concatenate([dgm1[:, 1], dgm2[:, 1]]) if len(dgm1) and len(dgm2) else \
             (dgm1[:, 1] if len(dgm1) else dgm2[:, 1])
    persistences = deaths - births
    if len(births) == 0:
        return 0.1, 0.1, 0.1
    max_birth = births.max()
    max_persistence = persistences.max()
    max_death = deaths.max()
    return max_birth, max_persistence, max_death

max_b, max_p, max_d = get_ranges(dgm_nominal, dgm_degraded)


max_b *= 1.1
max_p *= 1.1
max_d *= 1.1


max_b = max(max_b, 0.1)
max_p = max(max_p, 0.1)
max_d = max(max_d, 0.1)


res_img = 40
res_sil = 100

pi_nominal = persistence_image(dgm_nominal, resolution=(res_img, res_img),
                               sigma=0.02, max_b=max_b, max_p=max_p).reshape(res_img, res_img)
pi_degraded = persistence_image(dgm_degraded, resolution=(res_img, res_img),
                                sigma=0.02, max_b=max_b, max_p=max_p).reshape(res_img, res_img)

sil_nominal = persistence_silhouette(dgm_nominal, resolution=res_sil, max_grid=max_d)
sil_degraded = persistence_silhouette(dgm_degraded, resolution=res_sil, max_grid=max_d)
grid = np.linspace(0.0, max_d, res_sil)


fig, axs = plt.subplots(2, 2, figsize=(7.0, 5.5))


im0 = axs[0, 0].imshow(pi_nominal, origin="lower", cmap="inferno",
                       extent=[0, max_b, 0, max_p], aspect='auto')
axs[0, 0].set_title(r"Persistence Image ($\sigma_\tau = 0.0$)")
axs[0, 0].set_ylabel("Persistence")
cbar0 = fig.colorbar(im0, ax=axs[0, 0])
cbar0.ax.tick_params(labelsize=8)

im1 = axs[0, 1].imshow(pi_degraded, origin="lower", cmap="inferno",
                       extent=[0, max_b, 0, max_p], aspect='auto')
axs[0, 1].set_title(r"Persistence Image ($\sigma_\tau = 0.25$)")
cbar1 = fig.colorbar(im1, ax=axs[0, 1])
cbar1.ax.tick_params(labelsize=8)


axs[1, 0].fill_between(grid, sil_nominal, color='#003366', alpha=0.5)
axs[1, 0].plot(grid, sil_nominal, color='#003366', linewidth=1.5)
axs[1, 0].set_ylabel("Silhouette Height")
axs[1, 0].set_xlabel("Filtration Value")
axs[1, 0].set_xlim([0, max_d])

axs[1, 1].fill_between(grid, sil_degraded, color='#8B0000', alpha=0.5)
axs[1, 1].plot(grid, sil_degraded, color='#8B0000', linewidth=1.5)
axs[1, 1].set_xlabel("Filtration Value")
axs[1, 1].set_xlim([0, max_d])

plt.tight_layout()
plt.show()


fig = plt.figure(figsize=(7.0, 3.5))
axs = fig.subplots(1, 2)


im0 = axs[0].imshow(pi_nominal, origin="lower", cmap="inferno",
                    extent=[0, max_b, 0, max_p], aspect='auto')
axs[0].set_xlabel(r"$b$", fontsize=20)        
axs[0].set_ylabel(r"$p$", fontsize=20)        
axs[0].set_title(r" $\sigma_\tau = 0.0$", fontsize=16)
axs[0].tick_params(axis='both', which='major', labelsize=18)


axs[0].set_xticks([0, 0.05, 0.1])
axs[0].set_yticks([0, 0.05, 0.1])

cbar0 = fig.colorbar(im0, ax=axs[0])
cbar0.ax.tick_params(labelsize=18)  


im1 = axs[1].imshow(pi_degraded, origin="lower", cmap="inferno",
                    extent=[0, max_b, 0, max_p], aspect='auto')
axs[1].set_xlabel(r"$b$", fontsize=20)        
axs[1].set_ylabel(r"$p$", fontsize=20)        
axs[1].set_title(r" $\sigma_\tau = 0.25$", fontsize=16)
axs[1].tick_params(axis='both', which='major', labelsize=18)


axs[1].set_xticks([0, 0.05, 0.1])
axs[1].set_yticks([0, 0.05, 0.1])

cbar1 = fig.colorbar(im1, ax=axs[1])
cbar1.ax.tick_params(labelsize=18)

plt.tight_layout()
plt.show()


fig = plt.figure(figsize=(10, 5))
axs = fig.subplots(1, 2)


axs[0].fill_between(grid, sil_nominal, color='#003366', alpha=0.5)
axs[0].plot(grid, sil_nominal, color='#003366', linewidth=1.5)
axs[0].set_title(r" $\sigma_\tau = 0.0$", fontsize=26)
axs[0].set_xlabel(r"$x$", fontsize=26)               
axs[0].set_ylabel(r"$\Lambda(x)$", fontsize=26)     
axs[0].tick_params(axis='both', which='major', labelsize=26)
axs[0].set_xlim([0, max_d])


axs[1].fill_between(grid, sil_degraded, color='#8B0000', alpha=0.5)
axs[1].plot(grid, sil_degraded, color='#8B0000', linewidth=1.5)
axs[1].set_title(r" $\sigma_\tau = 0.25$", fontsize=26)
axs[1].set_xlabel(r"$x$", fontsize=26)               
axs[1].set_ylabel(r"$\Lambda(x)$", fontsize=26)     
axs[1].tick_params(axis='both', which='major', labelsize=26)
axs[1].set_xlim([0, max_d])

plt.tight_layout()
plt.show()

In [ ]:



def evaluate_classifier_cv(X_data, y_labels, feature_name):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000))
    ])
    
    
    cv_scores = cross_val_score(clf, X_data, y_labels, cv=5, scoring='roc_auc')
    print(f"{feature_name} CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    
    Xtr, Xte, ytr, yte = train_test_split(X_data, y_labels, test_size=0.30, stratify=y_labels, random_state=0)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    
    auc_single = roc_auc_score(yte, p)
    fpr, tpr, _ = roc_curve(yte, p)
    return fpr, tpr, auc_single


fpr_img, tpr_img, auc_img = evaluate_classifier_cv(X_img, y, "Persistence Images")


fpr_sil, tpr_sil, auc_sil = evaluate_classifier_cv(X_sil, y, "Silhouettes")


fpr_base, tpr_base, auc_base = evaluate_classifier_cv(X_base, y, "1D Baseline")


fig, ax = plt.subplots(figsize=(3.4, 2.8))

ax.plot(fpr_img, tpr_img, label=rf"Images (AUC = {auc_img:.3f})", color='#003366', linewidth=1.5)
ax.plot(fpr_sil, tpr_sil, label=rf"Silhouettes (AUC = {auc_sil:.3f})", color='#8B0000', linewidth=1.5)
ax.plot(fpr_base, tpr_base, label=rf"Baseline (AUC = {auc_base:.3f})", color='gray', linestyle='-.', linewidth=1.5)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, linewidth=1.0)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:



# Reuse the persistence-image bounds from dataset generation
fixed_max_b = 0.08
fixed_max_p = 0.08

n_pixels = X_img.shape[1]
res_img = int(np.sqrt(n_pixels))
pearson_map = np.zeros(n_pixels)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i in range(n_pixels):
        pixel_values = X_img[:, i]
        
        if np.std(pixel_values) == 0:
            pearson_map[i] = 0.0
        else:
            
            r, _ = pearsonr(sig_used, pixel_values)
            pearson_map[i] = r if not np.isnan(r) else 0.0

pearson_map_2d = pearson_map.reshape((res_img, res_img))

fig, ax = plt.subplots(figsize=(3.4, 2.8))

# Transpose so birth is horizontal and persistence is vertical
im = ax.imshow(pearson_map_2d.T, origin="lower", cmap="RdBu_r", 
               vmin=-1.0, vmax=1.0, extent=[0, fixed_max_b, 0, fixed_max_p], aspect='auto')

ax.set_xlabel("Birth")
ax.set_ylabel("Persistence")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r"Pearson $r$")

plt.tight_layout()
plt.show()